# Logarithmic Transformations & Quadratic Terms: Exercises
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> **Instructions:**
> - Work through the exercises in order; each builds on the previous one
> - Fill in your code in the cells marked with `# YOUR CODE HERE`
> - Answer written questions by double-clicking the markdown cell and editing it
> - Run cells with **Shift+Enter**
> - Solutions will be released after the submission deadline

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1: Name the Form (Warm-Up, No Code)

For each estimated equation, name the functional form (lin-lin, log-lin, lin-log, log-log) and interpret the highlighted coefficient in ONE precise sentence.

| # | Estimated equation | Interpret |
|---|--------------------|-----------|
| a | returns = 0.02 + **1.15**·mkt_returns | 1.15 |
| b | ln(price) = 4.60 **− 0.096**·yield | −0.096 |
| c | ln(volume) = 0.9 + **1.08**·ln(mcap) | 1.08 |
| d | salary = 20 + **0.45**·ln(AUM) | 0.45 (salary in kCHF) |
| e | ln(wage) = ... + **0.28**·D_CFA | 0.28 |

**Your answers** (double-click to edit):

| # | Form | One-sentence interpretation |
|---|------|-----------------------------|
| a | | |
| b | | |
| c | | |
| d | | |
| e | | |

---
# Data for Exercises 2 to 6: the Swiss Cross-Section

Run this provided cell once (it is the same loader as in the Lecture Notebook).

In [ ]:
# The course dataset for this chapter. It is a SIMULATED teaching cross-section,
# calibrated to the Swiss market - see the note below this cell.
CSV = ('https://raw.githubusercontent.com/KristynaTers/ASDA/main/data/'
       'ASDA_VolumeSize_Teaching_2026.csv')

raw = pd.read_csv(CSV, comment='#')
df = (raw.assign(mcap=raw['mcap_bn_chf'] * 1e9,
                 volume=raw['turnover_m_chf'] * 1e6)
         .set_index('firm_id')[['mcap', 'volume', 'D_SMI']].astype(float))

print('simulated teaching dataset - reproduces the numbers on the slides exactly')
print(f'n = {len(df)} firms ({int(df.D_SMI.sum())} index members, '
      f'{int((1 - df.D_SMI).sum())} mid caps)')
print(f'market cap  : {df["mcap"].min()/1e9:6.1f} to {df["mcap"].max()/1e9:6.1f} bn CHF'
      f'   (factor {df["mcap"].max()/df["mcap"].min():.0f})')
print(f'turnover    : {df["volume"].min()/1e6:6.1f} to {df["volume"].max()/1e6:6.1f} m CHF per day')

df['ln_vol']  = np.log(df['volume'])
df['ln_mcap'] = np.log(df['mcap'])


### About the dataset

The file loaded above is a **simulated teaching dataset**, not market data. The
firms are numbered (`CH01` … `CH50`), because they are not real companies. It is
one exact realisation of the data-generating process stated in this module's
Lecture Prep,

$$\ln(\text{turnover}) = 1.20 + 1.08\,\ln(\text{market cap}) + 0.35\,D_{SMI} + u,
\qquad n = 50$$

calibrated to the order of magnitude of the Swiss market (50 firms of roughly 3
to 50 bn CHF, 20 of them index members).

**Why simulated.** The lecture, the exercises and the solutions all quote the
same estimates, so the data behind them must be fixed. A live download from a
data provider changes every day, which would make the printed output impossible
to reproduce - and providers routinely drop or rename tickers. Loading this file
gives you the slides' numbers to the last digit:

| quantity | slides | this file |
|---|---|---|
| elasticity of turnover w.r.t. size | 1.08 (SE 0.07, t 15.4) | 1.08 (SE 0.07, t 15.4) |
| index-membership dummy | 0.35 (t 2.92) | 0.35 (t 2.92) |
| exact percentage effect | +41.9 % / −29.5 % | +41.9 % / −29.5 % |
| RESET, levels vs log-log | 18.3 vs 1.4 | 18.3 vs 1.4 |

**What the file is, precisely.** It is not a random draw: the residual vector is
constructed orthogonal to the regressors and then scaled, which is what makes
the estimates land on the stated coefficients and the standard errors on the
printed ones. In this sample, therefore, estimate and parameter coincide - on
real data they never do, and the exercises make that point separately.

**Two things the slides say that this sample does not carry.** The remark that
the same elasticity "answers the question for a 0.5 bn and for a 200 bn franc
company" is a statement about what a log-log coefficient *means*, not about the
sample's range, which is about 3 to 50 bn. And the named Swiss giants on the
scatter slide are there to make the levels-versus-logs picture concrete; the
firms in this file are numbered, not named.

If you want to repeat the exercise on live data, replace the loader with a
download of your own and expect different numbers - that is the point of the
distinction between an estimate and a parameter.


---
# Exercise 2: The Elasticity

Run the provided loader cell, then estimate the log-log model WITHOUT the dummy:

$$\ln V_i = \beta_0 + \beta_1 \ln M_i + u_i$$

Report the elasticity with a 95% confidence interval and interpret it in one sentence.

**Written question:** your neighbour estimated the same model on US stocks in dollars. Why are your two elasticities directly comparable even though the currencies differ?

In [ ]:
# YOUR CODE HERE
# X = sm.add_constant(df['ln_mcap']); OLS with HC1
# print beta_1, its 95% CI (m.conf_int()), one-sentence interpretation


---
# Exercise 3: The Dummy, Naive vs Exact

Add the SMI dummy and compute BOTH readings of its coefficient: the naive percentage (100·δ) and the exact percentage effect (100·(e^δ − 1)).

**Written question:** for which magnitudes of δ is the naive reading acceptable, and why exactly does it fail for large δ?

In [ ]:
# YOUR CODE HERE
# X = sm.add_constant(df[['ln_mcap', 'D_SMI']]); OLS with HC1
# delta = params['D_SMI']; naive = 100*delta; exact = 100*(np.exp(delta)-1)


---
# Exercise 4: Swap the Reference Category

Re-estimate the model with a MID-CAP dummy instead (D_MID = 1 − D_SMI).

(a) What is the new dummy coefficient, and how does it relate to the old one?
(b) Compute the exact percentage effect of being a mid cap. Explain why it is NOT simply minus the SMI effect.
(c) Verify numerically: (1 + g_up)·(1 + g_down) = 1, where g are the two exact effects as decimals.

In [ ]:
# YOUR CODE HERE
# df['D_MID'] = 1 - df['D_SMI']; re-estimate; compare coefficients and exact effects


---
# Exercise 5: RESET as Referee

Estimate the LEVELS specification (volume on mcap and the dummy, no logs) and run RESET on both the levels model and your log-log model from Exercise 3.

**Written question:** why would comparing the two R² values NOT be a valid way to choose between these models?

In [ ]:
# YOUR CODE HERE
# Scale first: mcap in bn and volume in m. In raw CHF the cubed fitted values reach 1e21,
# the auxiliary design goes rank-deficient and RESET silently tests ONE restriction
# instead of two (watch df_num).
# lvl = DataFrame(mcap_bn = mcap/1e9, vol_m = volume/1e6, D_SMI)
# m_lvl = OLS(vol_m on [mcap_bn, D_SMI]); linear_reset(..., power=3, use_f=True) on both models


---
# Exercise 6: A lin-log Variant

Regress volume IN MILLIONS (levels) on ln(mcap): a lin-log model. Interpret the slope in one sentence, being precise about units.

**Written question:** when might a lin-log form be economically more natural than log-log?

In [ ]:
# YOUR CODE HERE
# y = df['volume']/1e6; X = sm.add_constant(df['ln_mcap']); interpret slope/100


---
# Exercise 7: The Quadratic Fund Model

Load the same fund dataset the lecture uses, then estimate alpha on size and size squared with HC1 standard errors. Report both coefficients with t-statistics and state the shape.

```python
FUNDS_CSV = ('https://raw.githubusercontent.com/KristynaTers/ASDA/main/data/'
             'ASDA_FundSizeAlpha_Teaching_2026.csv')
funds = (pd.read_csv(FUNDS_CSV, comment='#')
           .rename(columns={'size_bn_chf': 'size', 'alpha_pct_pa': 'alpha'})
           .set_index('fund_id'))
```

In [ ]:
# YOUR CODE HERE
# build funds as above; funds['size2'] = size**2; OLS with HC1; report and interpret signs


---
# Exercise 8: Turning Point and Marginal Effects

(a) Compute the turning point x* = −β₁/(2β₂) and check whether it lies inside the data range.
(b) Compute the marginal effect β₁ + 2β₂x at sizes 0.2, at x*, and at 1.5 bn.
(c) Write ONE sentence a fund allocator could quote.

In [ ]:
# YOUR CODE HERE


---
# Exercise 9: What the Linear Model Would Have Told You

Fit the WRONG model, alpha on size only (no square), and run RESET on it.

**Written question:** what conclusion about fund size would the linear model suggest, why is it misleading, and how does RESET warn you?

In [ ]:
# YOUR CODE HERE
# m_lin = OLS(alpha on size); slope? RESET?


---
# Exercise 10: The Reporting Memo (Written)

In at most six sentences, write the results memo for both models: the elasticity and the SMI effect (with the exact percentage), the fund-size curve (marginal effects and turning point), and one sentence on why you chose these functional forms. Imagine the reader is a portfolio manager, not an econometrician.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*